In [1]:
!pip install "protobuf<7"

In [2]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

In [4]:
#Text data
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("data/python.txt", encoding="utf-8")

C:\Users\Saket\AppData\Local\Temp\ipykernel_10116\910593988.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [5]:
document = loader.load()

In [6]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [7]:
#pdf data

In [8]:
#from langchain_community.document_loaders.pdf import PyPDFLoader
#pdf_loader = PyPDFLoader("data/pdfs/research1.pdf")
#document = pdf_loader.load()

## Data Ingestion

In [9]:
#data => documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))

    return all_docs

In [10]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


## chunks

In [11]:
!pip install langchain_text_splitters

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap= chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs
    

In [13]:
chunks =split_docs(all_pdf_documents)

In [14]:
len(chunks)

321

## Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-V2"):
        self.model_name= model_name
        print("loading model ...", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())
    # to generate embedding
    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape:", embeddings.shape)
        return embeddings

In [17]:
embedding_manager = EmbeddingManager()

loading model ... all-MiniLM-L6-V2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\Saket\AppData\Local\Temp\ipykernel_10116\4049928347.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


## Vector Store

In [18]:
import chromadb
import uuid

In [19]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection =None
        self.client=None
        self._initialize_store()
    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        #create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection =self.client.get_or_create_collection(
            name =self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())
        
    #create a function to add all the documents
    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
        #store => ids, embedding, document, metadata
        ids =[]
        all_metadata =[]
        documents_content =[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            # metadata of individual metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] =i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            # now document 
            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )
        print("total documents added in vector store=",len(documents_content))
        print("docs in colections:", self.collection.count())
                
            

In [20]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [21]:
# chunks=> embedding
texts =[doc.page_content for doc in chunks]
embedding = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape: (321, 384)
total documents added in vector store= 321
docs in colections: 321


## Retrival pipelines implementation

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [24]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [25]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 4 documents


[{'id': 'doc_8bacfb4f-7c28-42e7-ae4c-020985cefd6f',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'page': 2,
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translati

## Integrate with LLMS OpenAI- GPT|

In [26]:
API_KEY_OPENAI = ""

In [27]:
!pip install langchain-openai

In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [29]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [ ]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

In [ ]:
print(answer)

### Groq

In [33]:
API_Key_GROQ = ""

In [34]:
!pip install langchain-groq

In [38]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

In [39]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [42]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 3 documents


In [43]:
print(answer)

**Encoder‑decoder** refers to the two‑part architecture used in Transformer models (and many other sequence‑to‑sequence systems).  

- **Encoder**: A stack of \(N = 6\) identical layers, each containing two sub‑layers:
  1. **Multi‑head self‑attention** – lets each token attend to every other token in the input sequence.
  2. **Feed‑forward network** – a simple position‑wise MLP.

  Residual connections and layer‑normalization wrap each sub‑layer, and dropout (\(P_{\text{drop}} = 0.1\)) is applied.

- **Decoder**: Also a stack of \(N = 6\) identical layers, but each layer has **three** sub‑layers:
  1. **Masked multi‑head self‑attention** – attends only to earlier positions in the output sequence.
  2. **Multi‑head attention over the encoder output** – allows the decoder to focus on relevant parts of the input.
  3. **Feed‑forward network** – same as in the encoder.

  Like the encoder, each sub‑layer is wrapped with residual connections, layer‑normalization, and dropout.

Together, th